### `nz_plots_polybias.ipynb`
---------------

Final comparison for the polynomial-bias variant: how the mean-redshift constraint moves
and how its width changes relative to the fiducial power-law analysis.

Unlike the magnification study there is a single realization here, so the comparison is
direct — fiducial posterior versus polynomial posterior, not a pooled mixture. The
relevant numbers are the shift in $\langle z \rangle$ (a bias) and the ratio of the two
$\sigma$ (a change in precision).

Needs the **`pymc`** kernel.

In [ ]:
import numpy as np
import pandas as pd
import importlib
import json
import matplotlib.pyplot as plt

from pathlib import Path

import src.statistics.spline as spline
import src.analysis.plots as plots
import src.statistics.corrfiles as cf
import src.statistics.systematics as sy

importlib.reload(spline)
importlib.reload(sy)

ROOT = cf.get_base_dir()

FIGURES_ROOT = ROOT / "paper" / "figures" / "systematics"
FIGURES_ROOT.mkdir(parents=True, exist_ok=True)
pm = plots.PlotManager(root=FIGURES_ROOT, overwrite=True)

In [ ]:
scale_cut = [0.3, 3]
version = "v_1p1"
names = ["npz_bs_bp", "npz_bs_bp_mag"]

tag = sy.scale_cut_tag(scale_cut)
DATA_DIR = sy.variant_dir(ROOT, "polybias", scale_cut, version)
SPL_DIR = sy.variant_dir(ROOT, "polybias", scale_cut, version, what="splines")
FID_SPL = ROOT / "results" / f"splines_{tag}_{version}"

with open(DATA_DIR / f"polybias_metadata_{tag}_{version}.json") as f:
    meta = json.load(f)

n_eval_points = 200
bin_colors = {1: "tab:blue", 2: "tab:orange", 3: "tab:red", 4: "tab:cyan"}
print(f"bias mode {meta['bias_mode']}, chi2/dof = "
      f"{meta['bias_chi2']:.1f}/{meta['bias_dof']}")

In [ ]:
summaries, rows = {}, []

for name in names:
    for tomo in sy.TOMO_BINS:
        f_fid = FID_SPL / f"spl_{name}_{tomo}"
        f_new = SPL_DIR / f"spl_{name}_{tomo}"
        for f in (f_fid, f_new):
            if not Path(f"{f}.nc").exists():
                raise FileNotFoundError(
                    f"{f}.nc missing -- run nz_splines.ipynb / "
                    "nz_splines_polybias.ipynb first."
                )

        spl_fid = spline.BayesianBSpline.from_saved_model(str(f_fid))
        spl_new = spline.BayesianBSpline.from_saved_model(str(f_new))
        z_eval = sy.common_grid([spl_fid, spl_new], n_points=n_eval_points)

        s_fid = sy.summarize_samples(
            sy.normalized_samples(spl_fid, z_eval, n_eval_points), z_eval
        )
        s_new = sy.summarize_samples(
            sy.normalized_samples(spl_new, z_eval, n_eval_points), z_eval
        )
        summaries[(name, tomo)] = (s_fid, s_new)

        rows.append(
            {
                "name": name,
                "tomo_bin": tomo,
                "mean_z_powerlaw": s_fid["mean_z"],
                "mean_z_polynomial": s_new["mean_z"],
                "delta_mean_z": s_new["mean_z"] - s_fid["mean_z"],
                "sigma_powerlaw": s_fid["mean_z_std"],
                "sigma_polynomial": s_new["mean_z_std"],
                "sigma_ratio": s_new["mean_z_std"] / s_fid["mean_z_std"],
                "mean_z_pl_p50": s_fid["mean_z_percentiles"][1],
                "mean_z_pl_lo": s_fid["mean_z_percentiles"][1] - s_fid["mean_z_percentiles"][0],
                "mean_z_pl_hi": s_fid["mean_z_percentiles"][2] - s_fid["mean_z_percentiles"][1],
                "mean_z_poly_p50": s_new["mean_z_percentiles"][1],
                "mean_z_poly_lo": s_new["mean_z_percentiles"][1] - s_new["mean_z_percentiles"][0],
                "mean_z_poly_hi": s_new["mean_z_percentiles"][2] - s_new["mean_z_percentiles"][1],
                "shift_in_sigma_fid": (
                    (s_new["mean_z"] - s_fid["mean_z"]) / s_fid["mean_z_std"]
                ),
            }
        )

budget = pd.DataFrame(rows).set_index(["name", "tomo_bin"])
budget

In [ ]:
print("Polynomial vs power-law photometric bias correction\n")
print(budget.to_string(float_format=lambda v: f"{v: .5f}"))
print(
    "\nsigma_ratio < 1 means the polynomial correction tightens the mean-redshift"
    "\nconstraint; shift_in_sigma_fid is the movement of <z> in units of the"
    "\nfiducial statistical error."
)

In [ ]:
name = "npz_bs_bp_mag"
with pm.make_plot(
    f"polybias_nz_bands_{tag}_{version}", figsize=(11, 7), nrows=2, ncols=2, show=True
) as (fig, axs):
    for tomo, ax in zip(sy.TOMO_BINS, axs.flat):
        s_fid, s_new = summaries[(name, tomo)]
        z = s_fid["z_eval"]

        ax.fill_between(z, s_fid["lower"], s_fid["upper"], color="0.5", alpha=0.4,
                        label="powerlaw")
        ax.fill_between(z, s_new["lower"], s_new["upper"], color=bin_colors[tomo],
                        alpha=0.45, label="polynomial")
        ax.plot(z, s_fid["median"], color="0.2", lw=1.6, ls="--")
        ax.plot(z, s_new["median"], color=bin_colors[tomo], lw=2)
        ax.axvline(s_fid["mean_z"], color="0.2", lw=1, ls="--")
        ax.axvline(s_new["mean_z"], color=bin_colors[tomo], lw=1)

        r = budget.loc[(name, tomo)]
        ax.set_title(
            f"Bin {tomo}  ($\\sigma$ ratio {r['sigma_ratio']:.3f}, "
            f"$\\Delta\\langle z\\rangle$ {r['delta_mean_z']:+.4f})"
        )
        ax.set_xlabel("Redshift ($z$)")
        ax.set_ylabel("$n(z)$")
        ax.grid(True, alpha=0.3)
        ax.axhline(0, color="black", ls="--", lw=1)
        if tomo == 1:
            ax.legend(fontsize=9)
    fig.suptitle("Magnification-corrected n(z): power-law vs polynomial bias correction")
    fig.tight_layout()

In [ ]:
with pm.make_plot(
    f"polybias_meanz_{tag}_{version}", figsize=(11, 7), nrows=2, ncols=2, show=True
) as (fig, axs):
    for tomo, ax in zip(sy.TOMO_BINS, axs.flat):
        s_fid, s_new = summaries[(name, tomo)]
        ax.hist(s_fid["mean_z_samples"], bins=60, density=True, color="0.5",
                alpha=0.55, label="powerlaw")
        ax.hist(s_new["mean_z_samples"], bins=60, density=True, histtype="step",
                color=bin_colors[tomo], lw=2, label="polynomial")
        r = budget.loc[(name, tomo)]
        ax.set_title(
            f"Bin {tomo}: $\\sigma$ {r['sigma_powerlaw']:.4f} $\\rightarrow$ "
            f"{r['sigma_polynomial']:.4f}"
        )
        ax.set_xlabel(r"$\langle z \rangle$")
        ax.set_ylabel("density")
        ax.grid(True, alpha=0.3)
        if tomo == 1:
            ax.legend(fontsize=9)
    fig.suptitle("Mean-redshift posterior, polynomial vs power-law bias correction")
    fig.tight_layout()

In [ ]:
out = {}
for (nm, tomo), (s_fid, s_new) in summaries.items():
    out[f"{nm}/{tomo}/z"] = s_fid["z_eval"]
    for label, s in (("powerlaw", s_fid), ("polynomial", s_new)):
        for key in ("median", "mean", "lower", "upper", "std"):
            out[f"{nm}/{tomo}/{label}_{key}"] = s[key]

np.savez_compressed(DATA_DIR / f"polybias_summary_{tag}_{version}.npz", **out)
budget.to_csv(DATA_DIR / f"polybias_sigma_budget_{tag}_{version}.csv")
print(f"Saved summary and sigma budget to {DATA_DIR}")